In [19]:
!pip install torch

In [11]:
import torch
%matplotlib inline
import hist
from hist import Hist
import coffea.processor as processor
import awkward as ak
from coffea.nanoevents import schemas

In [15]:
class Processor(processor.ProcessorABC):
    def __init__(self):
        dataset_axis = hist.cat('dataset','')
        MET_axis = hist.bin("MET","MET [GeV]", 50, 0, 100)

        self._accumulator = processor.dict_accumulator({'MET': hist.Hist("Counts", dataset_axis, MET_axis),
                                                       'Cutflow': processor.defaultdict_accumulator(int)})
    @property
    def accumulator(self):
        return self._accumulator

    def process(self, events):
        output = self.accumulator.identity()

        dataset = events.metadata["dataset"]
        MET = events.MET.pt

        output['cutflow']['all events'] += ak.size(MET)
        output['cutflow']['number of chunks'] += 1
        output['MET'].fill(dataset=dataset, MET=MET)

        return output

    def postprocess(self, accumulator):
        return accumulator

In [16]:
from dask.distributed import Client

client = Client("tls://localhost:8786")

In [17]:
fileset = {'SingleMu' : ["root://eospublic.cern.ch//eos/root-eos/benchmark/Run2012B_SingleMu.root"]}

executor = processor.DaskExecutor(client=client)

run = processor.Runner(executor=executor,
                        schema=schemas.NanoAODSchema,
                        savemetrics=True
                      )

output, metrics = run(fileset, "Events", processor_instance=Processor())

metrics

AttributeError: module hist has no attribute cat